# 🏦 Fundamental Valuation: DCF & Relative Valuation Guide

Welcome! This notebook provides an interactive quantitative guide to **Discounted Cash Flow (DCF)** valuation and **Relative Valuation (Comps)** using executable Python code and real financial data.

---

## 📌 Section 1: Discounted Cash Flow (DCF) Valuation

### 1. What is DCF?
**Discounted Cash Flow (DCF)** values a business based on the **present value of its projected future cash flows**. It answers: *"What is the intrinsic value of the business today based on all the cash it will generate in the future?"*

### 2. Formulas:
- **Present Value of Free Cash Flows (FCF):**
  
- **Terminal Value (TV) using Gordon Growth Model:**
  
- **Present Value of Terminal Value:**
  
- **Enterprise Value (EV):** 
- **Equity Value & Intrinsic Price Per Share:**
  
  

Where:
- **WACC:** Weighted Average Cost of Capital (Discount Rate / Required Return, e.g. 9%).
- **g:** Perpetual Terminal Growth Rate (typically 2%-3% matching long-term GDP growth).

---

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def dcf_valuation_yfinance(ticker_symbol="AAPL", fcf_growth=0.08, wacc=0.09, terminal_growth=0.025):
    print(f"Fetching real financial data for {ticker_symbol} from Yahoo Finance...")
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info
    cashflow = ticker.cashflow
    
    # 1. Extract Latest Free Cash Flow (Operating Cash Flow - Capital Expenditure)
    if "Free Cash Flow" in cashflow.index:
        latest_fcf = cashflow.loc["Free Cash Flow"].iloc[0]
    else:
        op_cashflow = cashflow.loc["Operating Cash Flow"].iloc[0]
        capex = abs(cashflow.loc["Capital Expenditure"].iloc[0])
        latest_fcf = op_cashflow - capex
        
    # 2. Extract Balance Sheet Data (Net Debt & Shares Outstanding)
    total_debt = info.get("totalDebt", 0)
    cash = info.get("totalCash", 0)
    net_debt = total_debt - cash
    shares_outstanding = info.get("sharesOutstanding", 1)
    current_price = info.get("currentPrice", info.get("previousClose", 0))
    
    # 3. Forecast 5 Years FCF & Discount to Present Value
    forecast_years = 5
    years = np.arange(1, forecast_years + 1)
    projected_fcf = [latest_fcf * ((1 + fcf_growth) ** t) for t in years]
    pv_fcf = [fcf / ((1 + wacc) ** t) for t, fcf in zip(years, projected_fcf)]
    
    # 4. Terminal Value (Gordon Growth Model)
    terminal_value = (projected_fcf[-1] * (1 + terminal_growth)) / (wacc - terminal_growth)
    pv_terminal_value = terminal_value / ((1 + wacc) ** forecast_years)
    
    # 5. Enterprise & Equity Intrinsic Value
    enterprise_value = sum(pv_fcf) + pv_terminal_value
    equity_value = enterprise_value - net_debt
    intrinsic_price = equity_value / shares_outstanding
    
    print(f"\n=== Real DCF Valuation: {ticker_symbol} ===")
    print(f"Latest FCF (Base Year):    ${latest_fcf/1e9:,.2f} Billion")
    print(f"Sum PV (5-Yr Forecast):   ${sum(pv_fcf)/1e9:,.2f} Billion")
    print(f"PV of Terminal Value:      ${pv_terminal_value/1e9:,.2f} Billion")
    print(f"Implied Enterprise Value:  ${enterprise_value/1e9:,.2f} Billion")
    print(f"Implied Equity Value:      ${equity_value/1e9:,.2f} Billion")
    print(f"Implied Intrinsic Price:   ${intrinsic_price:.2f}")
    print(f"Current Market Price:      ${current_price:.2f}")
    margin = ((intrinsic_price - current_price) / current_price) * 100
    print(f"Margin of Safety:          {margin:+.1f}%")
    
    return {
        "Ticker": ticker_symbol,
        "Intrinsic_Price": intrinsic_price,
        "Current_Price": current_price,
        "Projected_FCF": projected_fcf,
        "PV_FCF": pv_fcf
    }

# Run Real DCF for Apple (AAPL) and Alphabet (GOOGL)
aapl_dcf = dcf_valuation_yfinance("AAPL", fcf_growth=0.08, wacc=0.09)
googl_dcf = dcf_valuation_yfinance("GOOGL", fcf_growth=0.10, wacc=0.095)


## 📌 Section 2: Relative Valuation (Comparable Company Analysis / Comps)

### 1. What is Relative Valuation?
Instead of modeling intrinsic cash flows, **Relative Valuation** evaluates a target company by comparing its financial multiples against a peer group of similar companies.

### 2. Key Valuation Multiples:
- **P/E (Price-to-Earnings Ratio):** 
- **EV/EBITDA (Enterprise Value to EBITDA):**  (Capital-structure neutral metric).
- **P/S (Price-to-Sales Ratio):** 

### 3. Valuation Rule:
- If Target Company Multiple < Peer Median Multiple => **Undervalued (Bargain)**.
- If Target Company Multiple > Peer Median Multiple => **Overvalued (Expensive)**.

---

In [ ]:
def fetch_comps_matrix(tickers=["AAPL", "GOOGL", "MSFT", "AMZN", "META"]):
    print(f"Fetching real-time Comps peer data for {tickers} from Yahoo Finance...")
    comps_list = []
    
    for sym in tickers:
        t = yf.Ticker(sym)
        inf = t.info
        comps_list.append({
            "Ticker": sym,
            "Company": inf.get("shortName", sym),
            "PE_Forward": inf.get("forwardPE", np.nan),
            "PE_Trailing": inf.get("trailingPE", np.nan),
            "EV_EBITDA": inf.get("enterpriseToEbitda", np.nan),
            "Price_to_Sales": inf.get("priceToSalesTrailing12Months", np.nan)
        })
        
    df_real_comps = pd.DataFrame(comps_list)
    print("\n=== Real-Time Big Tech Relative Valuation Multiples ===")
    print(df_real_comps.to_string(index=False))
    
    # Plot Real P/E Multiples
    plt.figure(figsize=(11, 5))
    bars = plt.bar(df_real_comps["Ticker"], df_real_comps["PE_Forward"], color="#1f77b4", alpha=0.8)
    median_pe = df_real_comps["PE_Forward"].median()
    plt.axhline(median_pe, color="red", linestyle="--", linewidth=2, label=f"Peer Median Forward P/E ({median_pe:.1f}x)")
    plt.ylabel("Forward P/E Multiple (x)")
    plt.title("Real-Time Relative Valuation: Forward P/E Comparison", fontsize=13, fontweight="bold")
    plt.legend()
    plt.grid(True)
    plt.show()
    return df_real_comps

df_real_comps = fetch_comps_matrix()


--- 
## 📌 Section 3: Fundamental Multiples Deep-Dive (P/E, EV, EBITDA, P/S)

### 1. P/E (Price-to-Earnings Ratio)
- **What it is:** Measures how much investors pay for .00 of annual net earnings.
- **Formula:** P/E = Market Price Per Share / Earnings Per Share (EPS) = Market Capitalization / Net Income
  
- **Trailing vs. Forward P/E:**
  * **Trailing P/E:** Uses past 12 months (TTM) actual reported earnings.
  * **Forward P/E:** Uses consensus estimated earnings for the next 12 months.

---

### 2. EV (Enterprise Value)
- **What it is:** The true total takeover price of a company, representing value belonging to **both equity holders and debt holders**.
- **Formula:** Enterprise Value (EV) = Market Cap + Total Debt + Preferred Stock - Cash & Cash Equivalents
  
- **Why subtract Cash?** An acquirer can use the target company's cash balance to pay off part of the purchase price!

---

### 3. EBITDA (Earnings Before Interest, Taxes, Depreciation, & Amortization)
- **What it is:** Pure operational cash profitability before non-cash accounting adjustments and capital structure financing decisions.
- **Formula:** EBITDA = Net Income + Interest Expense + Taxes + Depreciation + Amortization
  
- **EV / EBITDA Multiple:**
  
  * **Why Quants Prefer EV/EBITDA over P/E:** P/E is distorted by different debt levels and tax rates. EV/EBITDA compares core operating performance cleanly across different countries and capital structures.

---

### 4. P/S (Price-to-Sales Ratio)
- **What it is:** Compares market value relative to top-line revenue.
- **Formula:** P/S = Market Capitalization / Total Annual Revenue = Share Price / Revenue Per Share
  
- **When to Use P/S:** Essential for valuing early-stage high-growth tech companies or startups that do not have positive net earnings yet.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Real Financial Breakdown Comparison across AAPL, GOOGL, MSFT
def analyze_financial_multiples(tickers=["AAPL", "GOOGL", "MSFT"]):
    data = []
    for sym in tickers:
        inf = yf.Ticker(sym).info
        mcap = inf.get("marketCap", 0) / 1e9
        ev = inf.get("enterpriseValue", 0) / 1e9
        ebitda = inf.get("ebitda", 0) / 1e9
        rev = inf.get("totalRevenue", 0) / 1e9
        pe_t = inf.get("trailingPE", np.nan)
        pe_f = inf.get("forwardPE", np.nan)
        ev_ebitda = inf.get("enterpriseToEbitda", np.nan)
        ps = inf.get("priceToSalesTrailing12Months", np.nan)
        
        data.append({
            "Ticker": sym,
            "MarketCap_()": mcap,
            "EV_()": ev,
            "EBITDA_()": ebitda,
            "Revenue_()": rev,
            "Trailing_PE": pe_t,
            "Forward_PE": pe_f,
            "EV_EBITDA": ev_ebitda,
            "P_S": ps
        })
        
    df_m = pd.DataFrame(data)
    print("=== Section 3: Fundamental Valuation Multiples Breakdown ===")
    print(df_m.to_string(index=False))
    
    # Plot Multi-Metric Comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(tickers))
    width = 0.35
    ax1.bar(x - width/2, df_m["Trailing_PE"], width, label="Trailing P/E", color="#1f77b4")
    ax1.bar(x + width/2, df_m["Forward_PE"], width, label="Forward P/E", color="#2ca02c")
    ax1.set_xticks(x)
    ax1.set_xticklabels(tickers)
    ax1.set_ylabel("Multiple (x)")
    ax1.set_title("Trailing vs. Forward P/E Ratio", fontsize=12, fontweight="bold")
    ax1.legend()
    ax1.grid(True)
    
    ax2.bar(df_m["Ticker"], df_m["EV_EBITDA"], color="#ff7f0e", alpha=0.8)
    ax2.set_ylabel("EV / EBITDA (x)")
    ax2.set_title("Enterprise Multiple (EV / EBITDA)", fontsize=12, fontweight="bold")
    ax2.grid(True)
    plt.tight_layout()
    plt.show()
    return df_m

df_m = analyze_financial_multiples()
